# Challenge Data Generation

*Decision Academy — Rethinking How We Choose Statistical Tests*

This notebook generates `challenge_data.csv` with a known ground truth, so we can verify which statistical tests recover the true effect.

**Design:**
- Control group (n=50): Normal(50, 10)
- Treatment group (n=50): bimodal mixture — 55% non-responders Normal(42, 8) + 45% responders Normal(78, 8)
- True treatment mean: 0.55 × 42 + 0.45 × 78 = 58.2 → **true effect = +8.2**
- Site effects: Site A (+0), Site B (−3), Site C (+2)
- Seed chosen (394) so that Welch's t-test detects the effect (p=0.033), Mann-Whitney misses it (p=0.40), and Shapiro-Wilk rejects normality (p=0.003)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(394)

N_PER_GROUP = 50
N_TOTAL = N_PER_GROUP * 2

## 1. Generate Outcome Scores

In [ ]:
# Control group: normal
ctrl_scores = np.random.normal(50, 10, N_PER_GROUP)

# Treatment group: bimodal mixture
n_nonresp = int(N_PER_GROUP * 0.55)  # 27 non-responders
n_resp = N_PER_GROUP - n_nonresp      # 23 responders

trt_nonresp = np.random.normal(42, 8, n_nonresp)
trt_resp = np.random.normal(78, 8, n_resp)
trt_scores = np.concatenate([trt_nonresp, trt_resp])

# Shuffle treatment so responders aren't at the end
trt_scores = trt_scores[np.random.permutation(N_PER_GROUP)]

print(f"True treatment mean: {0.55 * 42 + 0.45 * 78:.1f}")
print(f"True effect: +{0.55 * 42 + 0.45 * 78 - 50:.1f}")
print(f"Observed control mean: {ctrl_scores.mean():.2f}")
print(f"Observed treatment mean: {trt_scores.mean():.2f}")
print(f"Observed difference: {trt_scores.mean() - ctrl_scores.mean():.2f}")

## 2. Generate Covariates

In [ ]:
# Age: roughly balanced between groups
ctrl_age = np.random.normal(45, 12, N_PER_GROUP).clip(20, 75).astype(int)
trt_age = np.random.normal(45, 12, N_PER_GROUP).clip(20, 75).astype(int)

# Baseline cognitive score: correlated with outcome for control,
# independent of treatment mixture (measured before intervention)
ctrl_baseline = ctrl_scores * 0.6 + np.random.normal(30, 8, N_PER_GROUP)
trt_baseline = np.random.normal(50, 10, N_PER_GROUP) * 0.6 + np.random.normal(30, 8, N_PER_GROUP)

# Site: 3 study sites with different baseline levels
sites = np.random.choice(['Site_A', 'Site_B', 'Site_C'], N_TOTAL, p=[0.4, 0.35, 0.25])
site_effects = {'Site_A': 0, 'Site_B': -3, 'Site_C': 2}

print(f"Site distribution: {dict(zip(*np.unique(sites, return_counts=True)))}")

## 3. Apply Site Effects and Build DataFrame

In [ ]:
# Apply site effects to scores
all_scores = np.concatenate([ctrl_scores, trt_scores])
for i in range(N_TOTAL):
    all_scores[i] += site_effects[sites[i]]

ctrl_scores = all_scores[:N_PER_GROUP]
trt_scores = all_scores[N_PER_GROUP:]

# Build dataframe
df = pd.DataFrame({
    'participant_id': [f"P{str(i+1).zfill(3)}" for i in range(N_TOTAL)],
    'group': ['control'] * N_PER_GROUP + ['treatment'] * N_PER_GROUP,
    'site': sites,
    'age': np.concatenate([ctrl_age, trt_age]),
    'baseline_score': np.round(np.concatenate([ctrl_baseline, trt_baseline]), 1),
    'cognitive_score': np.round(all_scores, 1)
})

# Shuffle so groups aren't in order
df = df.sample(frac=1, random_state=394).reset_index(drop=True)

print(f"Shape: {df.shape}")
df.head(10)

## 4. Verify the Design Works

In [ ]:
ctrl = df[df['group'] == 'control']['cognitive_score']
trt = df[df['group'] == 'treatment']['cognitive_score']

sw_trt = stats.shapiro(trt)
welch = stats.ttest_ind(trt, ctrl, equal_var=False)
mw = stats.mannwhitneyu(trt, ctrl, alternative='two-sided')

print("=== Verification ===")
print(f"Shapiro-Wilk (treatment): W={sw_trt.statistic:.4f}, p={sw_trt.pvalue:.6f}  {'REJECTS normality' if sw_trt.pvalue < 0.05 else ''}")
print(f"Welch t-test:             t={welch.statistic:.4f}, p={welch.pvalue:.4f}      {'SIGNIFICANT' if welch.pvalue < 0.05 else 'not significant'}")
print(f"Mann-Whitney U:           U={mw.statistic:.1f},  p={mw.pvalue:.4f}      {'SIGNIFICANT' if mw.pvalue < 0.05 else 'NOT significant'}")
print()
print(f"Observed mean difference: {trt.mean() - ctrl.mean():.2f}")
print(f"True effect: +8.2")
print()

ok = sw_trt.pvalue < 0.05 and welch.pvalue < 0.05 and mw.pvalue > 0.05
print(f"Design works as intended: {ok}")
if ok:
    print("  - Shapiro-Wilk rejects normality for treatment")
    print("  - Welch's t-test detects the real effect")
    print("  - Mann-Whitney U misses it")

## 5. Save

In [ ]:
df.to_csv('challenge_data.csv', index=False)
print(f"Saved challenge_data.csv ({df.shape[0]} rows, {df.shape[1]} columns)")